# Prompt Playground

Scratchpad for writing and testing Claude prompts before wiring them into an app.
Edit the prompt cells below and re-run — no need to restart the kernel between iterations.

## Setup (run once)

1. Install deps: `pip install -r requirements.txt` (from this `notebooks/` folder), or run the cell below.
2. Copy `.env.example` to `.env` in this folder and paste your key from https://console.anthropic.com — `.env` is gitignored.
3. Run the cells top to bottom once, then jump back to **Write your prompt** and iterate freely.

In [ ]:
# One-time install — safe to re-run, no-ops if already installed.
%pip install -q anthropic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from notebooks/.env

client = anthropic.Anthropic()

MODEL = "claude-opus-5"
EFFORT = "medium"  # low | medium | high | xhigh | max — raise for harder prompts
SHOW_THINKING = True  # print Claude's reasoning summary alongside the answer

## Helper: `ask()`

Streams the response so long prompts never hit an HTTP timeout, prints as it goes, and
returns the final message (so you can inspect `usage`, `stop_reason`, etc. after).

In [ ]:
def ask(system: str, user: str, model: str = MODEL, effort: str = EFFORT, show_thinking: bool = SHOW_THINKING):
    """Send one prompt to Claude and stream the response into this cell's output."""
    with client.messages.stream(
        model=model,
        max_tokens=8192,
        thinking={"type": "adaptive", "display": "summarized" if show_thinking else "omitted"},
        output_config={"effort": effort},
        system=system,
        messages=[{"role": "user", "content": user}],
    ) as stream:
        printed_thinking_header = False
        printed_text_header = False
        for event in stream:
            if event.type == "content_block_delta":
                if event.delta.type == "thinking_delta" and show_thinking:
                    if not printed_thinking_header:
                        print("--- thinking " + "-" * 40)
                        printed_thinking_header = True
                    print(event.delta.thinking, end="", flush=True)
                elif event.delta.type == "text_delta":
                    if not printed_text_header:
                        print("\n--- response " + "-" * 40)
                        printed_text_header = True
                    print(event.delta.text, end="", flush=True)
        response = stream.get_final_message()

    print("\n\n--- usage " + "-" * 40)
    print(
        f"stop_reason={response.stop_reason}  "
        f"input_tokens={response.usage.input_tokens}  "
        f"output_tokens={response.usage.output_tokens}"
    )
    return response

## Write your prompt

Edit `SYSTEM_PROMPT` / `USER_PROMPT` and re-run this cell and the one below as many
times as you like. Nothing here is tied to a specific app — swap in whatever you're testing.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."

USER_PROMPT = "Say hello and confirm the playground is working."

In [ ]:
response = ask(SYSTEM_PROMPT, USER_PROMPT)

## Compare prompt variants side by side

Useful once you have two or three candidate phrasings and want to see outputs together
instead of scrolling back through cell history. Each variant is a `(label, system, user)` tuple.

In [ ]:
def compare(variants: list[tuple[str, str, str]], **ask_kwargs):
    """variants: list of (label, system_prompt, user_prompt)."""
    results = {}
    for label, system, user in variants:
        print(f"\n{'=' * 60}\n{label}\n{'=' * 60}")
        results[label] = ask(system, user, show_thinking=False, **ask_kwargs)
    return results


# Example:
# compare([
#     ("terse", "Answer in one sentence.", USER_PROMPT),
#     ("detailed", "Answer thoroughly with examples.", USER_PROMPT),
# ])

## Next: move into the app

Once a prompt earns its place, copy `SYSTEM_PROMPT` / `USER_PROMPT` (and the `model` /
`effort` you settled on) into the app code. This notebook and its `.env` are gitignored
scratch space — they don't ship with the app.